## ***Data Science Phase 2 Task 4***

***Loan Approval Prediction System***

In [53]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [54]:
# 1. Load Training Dataset (Use train_loan.csv because test_loan.csv lacks target labels)
df = pd.read_csv('train_loan.csv')

In [55]:
# 2. Data Cleaning & Preprocessing
if 'Loan_ID' in df.columns:
    df = df.drop(columns=['Loan_ID'])

# Handle missing values: Fill numerical with median, categorical with mode
for col in df.select_dtypes(include=[np.number]).columns:
    df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include=[object]).columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

/tmp/ipykernel_2582/490284266.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_2582/490284266.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usi

In [56]:
# 3. Feature Engineering (To boost model performance)
if 'ApplicantIncome' in df.columns and 'CoapplicantIncome' in df.columns:
    df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
    df['Log_Total_Income'] = np.log(df['Total_Income'] + 1)

if 'LoanAmount' in df.columns and 'Total_Income' in df.columns:
    df['Income_Loan_Ratio'] = df['Total_Income'] / (df['LoanAmount'] * 1000 + 1)

In [57]:
# Encode categorical variables
for col in df.select_dtypes(include=[object]).columns:
    if col != 'Loan_Status':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

# Encode target variable 'Loan_Status' (Y/N to 1/0)
if 'Loan_Status' in df.columns:
    df['Loan_Status'] = df['Loan_Status'].apply(lambda x: 1 if x == 'Y' or x == 1 else 0)

In [58]:
# Define Features (X) and Target (y)
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [59]:
# 4. Model Training & Testing Split (As required in Task 4)[cite: 1]
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

In [60]:
# 5. Define Models with Class Balancing
models = {
    "Logistic Regression": LogisticRegression(random_state=42, class_weight='balanced'),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=5),
    "Random Forest Classifier": RandomForestClassifier(random_state=42, class_weight='balanced', n_estimators=100)
}

In [61]:
# 6. Train and Evaluate Models
evaluation_results = []

for name, model in models.items():
    # Train the model[cite: 1]
    model.fit(X_train, y_train)

    # Predict on test set
    y_pred = model.predict(X_test)

    # Calculate performance evaluation metrics[cite: 1]
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    evaluation_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    print(f"--- {name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}\n")

--- Logistic Regression ---
Accuracy:  0.8049
Precision: 0.8427
Recall:    0.8824
F1 Score:  0.8621
Confusion Matrix:
[[24 14]
 [10 75]]

--- Decision Tree Classifier ---
Accuracy:  0.8211
Precision: 0.8462
Recall:    0.9059
F1 Score:  0.8750
Confusion Matrix:
[[24 14]
 [ 8 77]]

--- Random Forest Classifier ---
Accuracy:  0.8618
Precision: 0.8542
Recall:    0.9647
F1 Score:  0.9061
Confusion Matrix:
[[24 14]
 [ 3 82]]



In [62]:
# 7. Summary Comparison Table
results_df = pd.DataFrame(evaluation_results)
print("=== Model Performance Summary ===")
print(results_df.to_string(index=False))

=== Model Performance Summary ===
                   Model  Accuracy  Precision   Recall  F1 Score
     Logistic Regression  0.804878   0.842697 0.882353  0.862069
Decision Tree Classifier  0.821138   0.846154 0.905882  0.875000
Random Forest Classifier  0.861789   0.854167 0.964706  0.906077
